In [4]:
import numpy as np
import cv2
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

# Sample annotation (normalized coordinates)
annotations = [0, 0.590734, 0.551546, 0.648649, 0.257732, 0.915058, 0.603093, 
               0.710425, 0.572165, 0.625483, 0.515464, 0.833977, 0.525773, 
               0.714286, 0.448454, 0.416988, 0.422680, 0.266409, 0.680412, 
               0.308880, 0.567010]

# Convert annotations to list of coordinates (normalized)
keypoints = [(annotations[i], annotations[i+1]) for i in range(1, len(annotations), 2)]

# Function to adjust keypoints based on transformations
def adjust_keypoints(keypoints, transformation, image_size):
    adjusted_keypoints = []
    for x, y in keypoints:
        if 'scale' in transformation:
            x, y = x * transformation['scale'], y * transformation['scale']
        if 'translate' in transformation:
            x, y = x + transformation['translate'][0], y + transformation['translate'][1]
        if 'rotate' in transformation:
            angle = np.radians(transformation['rotate'])
            center_x, center_y = image_size[0] / 2, image_size[1] / 2
            x, y = x - center_x, y - center_y
            x_new = x * np.cos(angle) - y * np.sin(angle)
            y_new = x * np.sin(angle) + y * np.cos(angle)
            x, y = x_new + center_x, y_new + center_y
        if 'flip' in transformation:
            if transformation['flip'] == 'horizontal':
                x = 1 - x  # Flip x-axis (normalized)
            elif transformation['flip'] == 'vertical':
                y = 1 - y  # Flip y-axis (normalized)
        adjusted_keypoints.append((x, y))
    return adjusted_keypoints

# Load image
img = cv2.imread('image.png')
img = cv2.resize(img, (200, 200))

# Apply the augmentation using ImageDataGenerator
datagen = ImageDataGenerator(
    rotation_range=20,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2,
)

img_array = img.reshape(1, 200, 200, 3)

# Define the save directory for augmented images and keypoints
save_dir = "/workspaces/CNN/4.data augmentation/aug"
os.makedirs(save_dir, exist_ok=True)

# Initialize counter
i = 0

# Loop to generate augmented images and save their annotations
for batch in datagen.flow(img_array, batch_size=1, save_to_dir=save_dir):
    # Define the transformation applied to the image (you can adjust this for each image)
    transformation = {
        'scale': 1.2, 
        'translate': (0.05, -0.05),
        'rotate': 30,  # in degrees
        'flip': 'horizontal'  # Example of flip
    }
    
    # Adjust keypoints for the current augmentation
    adjusted_keypoints = adjust_keypoints(keypoints, transformation, img.shape[:2])

    # Save the augmented image
    augmented_image_path = os.path.join(save_dir, f"augmented_{i}.png")
    
    # Save the adjusted keypoints in a text file
    augmented_keypoints_path = os.path.join(save_dir, f"augmented_{i}_keypoints.txt")
    with open(augmented_keypoints_path, 'w') as f:
        f.write("0 " + " ".join([f"{x} {y}" for x, y in adjusted_keypoints]))
    
    # Increment counter
    i += 1
    if i == 10:  # Stop after generating 10 images
        break

